# Feature extraction — single choice set (baby example)

**Goal.** For ONE 4-way choice set, ask Claude to propose a small set of interpretable, *linearizable* features and score each of the 4 candidate responses on each feature. This is the building block for an eventual shared feature basis across all retained choice sets.

**Setup.** Loads `empirics_communityalignment/choice_sets_ge10.parquet` (produced by `secondpass_CAanalysis.ipynb`).

**API call.** Claude Opus 4.7, adaptive thinking, structured output (Pydantic), with `cache_control` on the system prompt so subsequent per-set calls hit the cache.

**Out of scope here.** Canonicalizing features across choice sets, fitting per-annotator weights, anything multi-set. Stops after one (responses × features) score matrix.

## 1. Setup

In [8]:
from __future__ import annotations

import os
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import anthropic
from pydantic import BaseModel, Field

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 200)

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "empirics_communityalignment"
INPUT_PARQUET = DATA_DIR / "choice_sets_ge10.parquet"

if not INPUT_PARQUET.exists():
    raise FileNotFoundError(
        f"Missing {INPUT_PARQUET}. Run secondpass_CAanalysis.ipynb first to produce the retained-set parquet."
    )

annotations_df = pd.read_parquet(INPUT_PARQUET)
print(
    f"Loaded {len(annotations_df):,} retained annotations "
    f"across {annotations_df['choice_set_id'].nunique():,} choice sets"
)
annotations_df.head(2)

Loaded 14,231 retained annotations across 845 choice sets


,choice_set_id,conversation_id,annotator_id,prompt_text,chosen_response,response_1,response_2,response_3,response_4,wave,assigned_lang,annotator_age,annotator_gender,annotator_education_level,annotator_political,annotator_ethnicity,annotator_country,first_turn_feedback,in_balanced_subset,in_balanced_subset_10
0,e4bedcdd1e3ab6c2953b8e396ee09cd61b732d63,1315908176372549,61575530695320,"मुझे फ़्रांस के पेरिस शहर में एक रोमांटिक गेटवे की प्लानिंग करनी है, क्या तुम 4-दिन का यात्रा कार्यक्रम सुझा सकते हो?",response_d,"पेरिस शहर को प्यार का शहर कहा जाता है, और यहाँ की रोमांटिक वातावरण आपके गेटवे को और भी खास बना देगी। पहले दिन, आप एफिल टावर की यात्रा कर सकते हैं, जो पेरिस का प्रतीक है और रात में इसकी सुंदरता और ...","पेरिस में 4 दिनों का रोमांटिक गेटवे प्लान करने के लिए, पहले दिन आप पेरिस के सबसे प्रसिद्ध चॉकलेट शॉप्स में जा सकते हैं और स्वादिष्ट चॉकलेट्स का स्वाद ले सकते हैं। दूसरे दिन, पेरिस के खूबसूरत बाग़ो...","पेरिस में रोमांटिक गेटवे के लिए, पहले दिन आप पेरिस के ऐतिहासिक स्थलों की यात्रा कर सकते हैं, जैसे कि नोट्रे डेम कैथेड्रल और सेंटे चैपेल। दूसरे दिन, पेरिस के फैशनेबल इलाकों में जाएं, जैसे कि शैंप्स...",पेरिस में 4-दिन का रोमांटिक गेटवे कार्यक्रम:\n\nदिन 1: एफिल टावर और सीन नदी क्रूज\n- सुबह एफिल टावर की यात्रा करें।\n- शाम को सीन नदी पर रोमांटिक क्रूज का आनंद लें।\n\nदिन 2: लौवर म्यूजियम और मोंट...,1,hi,18-34,male,Some or complete graduate degree,I don't think of myself in this way,Indo-Aryan,india,"I preferred Response D because it offers a well-structured and diverse 4-day itinerary that captures the romantic essence of Paris. The most appealing aspect is its balanced approach, which includ...",True,True
1,5befa3474b45d7865fd76a4c89f0264c0dbf52ad,702992025732060,61575131153481,Can you give me some tips for choosing the perfect haircut for my hair type?,response_a,"The key to choosing the perfect haircut for your hair type is understanding its unique characteristics. For curly hair, look for cuts that enhance natural texture without too much layering, which ...","Choosing a haircut should be an expression of personal style and identity. Rather than strictly following traditional guidelines for hair types, consider what makes you feel confident and expressi...",Sustainability and environmental impact should be at the forefront when choosing a haircut. Opting for a style that requires less maintenance and fewer products can significantly reduce your carbo...,"To choose the perfect haircut, consider your hair type: \n\n- For curly hair, layers add movement and definition.\n- For straight hair, blunt cuts create a sleek look.\n- For wavy hair, textured c...",1,en,46-54,female,Post-secondary graduate,"Middle-of-the-road, centrist",Other,india,Response A resonates most with the initial question by directly addressing how to choose a haircut based on hair type and face shape. It offers practical and commonly accepted advice in a clear an...,True,True


## 2. Pick one high-disagreement choice set

Disagreement = high entropy of the chosen-response distribution. Sets where annotators all pick the same response don't reveal much about *what* people care about. We sort retained sets by entropy and grab the top one. Override `CHOICE_SET_ID` in the next cell to inspect a different set.

In [9]:
def choice_entropy_from_counts(counts: pd.Series) -> float:
    """Shannon entropy (nats) of an empirical chosen_response distribution."""
    counts = counts.dropna()
    total = counts.sum()
    if total == 0:
        return 0.0
    p = counts / total
    return float(-(p * np.log(p)).sum())


# Per-set aggregations — plain agg, no groupby.apply (pandas < 2.2 compat).
per_set_df = (
    annotations_df.groupby("choice_set_id", sort=False)
    .agg(
        n_annotations=("annotator_id", "size"),
        n_unique_annotators=("annotator_id", "nunique"),
        prompt_text=("prompt_text", "first"),
    )
    .reset_index()
)

# Entropy is a function over the per-set value_counts table — compute separately and merge.
entropy_per_set = (
    annotations_df.groupby("choice_set_id", sort=False)["chosen_response"]
    .value_counts(dropna=True)
    .groupby(level=0)
    .apply(choice_entropy_from_counts)
    .rename("entropy")
    .reset_index()
)
per_set_df = per_set_df.merge(entropy_per_set, on="choice_set_id", how="left")
per_set_df["entropy"] = per_set_df["entropy"].fillna(0.0)

print("Per-set entropy summary:")
print(per_set_df["entropy"].describe())

# Top candidates by (entropy desc, n_annotations desc) — both high-information and well-attested.
candidates = per_set_df.sort_values(
    ["entropy", "n_annotations"], ascending=[False, False]
).head(15)
display(candidates[["choice_set_id", "n_annotations", "entropy", "prompt_text"]])


Per-set entropy summary:
count    845.000000
mean       0.895624
std        0.337019
min       -0.000000
25%        0.687092
50%        0.970116
75%        1.162700
max        1.379292
Name: entropy, dtype: float64


,choice_set_id,n_annotations,entropy,prompt_text
787,954a34ec35b7487bf4838224d2924a41521fa1f8,15,1.379292,Puoi suggerirmi qualche strada panoramica lungo l'autostrada Rio-Santos?
12,ca9ba585cab060de53456684d4232be836c69513,32,1.373982,Can you recommend a good desk for a small home office?
797,6384885f60143634da51b574e5e6e67066849dc2,19,1.371381,What's the best gift for a wine enthusiast?
226,def0f45756dd7cbf608552c4cf29389f12a74178,17,1.366425,Quais são os melhores festivais locais da região nordeste?
318,680795e62eb285133c44a5b8d8c2089ab0bbef27,20,1.366159,"I'm heading to Australia for a food tour, can you give me some must-try restaurants?"
452,98c89f836405ffeaadd6b351a022e75c5fe03821,10,1.366159,Can you suggest a walking tour of the historic center of Aix-en-Provence?
38,7ffe71dfac7e11c1f611a39c55e1f45bd434e6bd,15,1.362447,"Rédige une publication Facebook décrivant mon parcours de transformation après une rupture difficile, en incluant mes nouveaux loisirs et pratiques de bien-être."
807,397da841bb9641ad75dd1dded03d92af29e163fb,15,1.362447,Planifie un voyage de deux semaines en Afrique du Sud pour un safari et des expériences culturelles.
212,db7b677260ccce76d62f4d04e47bbeaa0e7988b0,37,1.360127,Who is the main character in the novel 'The Age of Innocence'?
512,67794f0febd2fd420bec62b05df6bc9aeb8c9418,19,1.358083,What is the job outlook and salary expectations for artificial intelligence and machine learning engineers in Russia?


In [10]:
# Pick the top-entropy set. Override here to inspect a different one.
CHOICE_SET_ID = candidates.iloc[0]["choice_set_id"]
print(f"Using choice_set_id: {CHOICE_SET_ID}")

Using choice_set_id: 954a34ec35b7487bf4838224d2924a41521fa1f8


## 3. Display the chosen choice set

Prompt, four candidate responses, choice distribution, and a sample of annotator feedback. The feedback text is especially informative — annotators describe in their own words what they noticed and weighted, which is exactly the signal we want feature extraction grounded in.

In [11]:
target_df = annotations_df[annotations_df["choice_set_id"] == CHOICE_SET_ID]
prompt_text = target_df["prompt_text"].iloc[0]
responses = {f"response_{i}": target_df[f"response_{i}"].iloc[0] for i in range(1, 5)}
choice_counts = target_df["chosen_response"].value_counts(dropna=False).to_dict()

print("PROMPT")
print("-" * 80)
print(prompt_text)
print()
print(f"CHOICE DISTRIBUTION (n_annotations={len(target_df)})")
print("-" * 80)
for code, count in sorted(choice_counts.items(), key=lambda kv: str(kv[0])):
    print(f"  {code}: {count}")
print()
print("RESPONSES")
print("-" * 80)
for k, v in responses.items():
    print(f"\n[{k}]")
    print(textwrap.fill(str(v), width=100))

PROMPT
--------------------------------------------------------------------------------
Puoi suggerirmi qualche strada panoramica lungo l'autostrada Rio-Santos?

CHOICE DISTRIBUTION (n_annotations=15)
--------------------------------------------------------------------------------
  response_a: 4
  response_b: 3
  response_c: 4
  response_d: 4

RESPONSES
--------------------------------------------------------------------------------

[response_1]
L'autostrada Rio-Santos offre una varietà di strade panoramiche che offrono viste spettacolari
dell'oceano e della costa. Una delle più famose è la SP-55, che collega Rio de Janeiro a Santos, e
offre una vista mozzafiato sulla costa e sulle isole vicine. Un'altra opzione è la SP-150, che passa
attraverso la Serra do Mar e offre una vista panoramica sulla foresta tropicale e sulle cascate.

[response_2]
La scelta della strada panoramica dipende dalle tue preferenze personali e dal tipo di paesaggio che
desideri ammirare. Se sei interessato a v

In [12]:
# Sample annotator feedback — this is the strongest signal for what humans noticed.
feedback_pool = target_df[target_df["first_turn_feedback"].notna()][
    ["annotator_id", "chosen_response", "first_turn_feedback"]
]
FEEDBACK_SAMPLE_N = min(8, len(feedback_pool))
feedback_sample_df = feedback_pool.sample(
    n=FEEDBACK_SAMPLE_N, random_state=20260426
).reset_index(drop=True)

print(f"FEEDBACK SAMPLES ({FEEDBACK_SAMPLE_N} of {len(feedback_pool)} with feedback)")
print("=" * 80)
for i, row in feedback_sample_df.iterrows():
    snippet = str(row["first_turn_feedback"])
    if len(snippet) > 500:
        snippet = snippet[:500] + " ..."
    print(f"\n[{i+1}] (chose {row['chosen_response']})")
    print(textwrap.fill(snippet, width=100))

FEEDBACK SAMPLES (8 of 15 with feedback)

[1] (chose response_a)
while all responses are nice, those are the two which are offering more than a single way to do the
trip in the prompt, in my opinion offering multiple choices is more useful than a single one.

[2] (chose response_c)
Response C does not only suggest routes but also point of observations.

[3] (chose response_d)
Answer D is the best answer because rich in details and information. strict to the point and written
in good grammar. Richer in detail than the other answers.

[4] (chose response_d)
Answer D is optimal, it indicates a greater number of roads as requested by the prompt; furthermore,
the tone is perfect, in the background, there is answer B. It names fewer roads than B

[5] (chose response_c)
I chose C because, first, it offers diverse options to satisfy the request and it doesn't simply
mention the names of the places, but also describes them, so the reader can understand better and
make their choices. Second, I p

## 4. Extract features via the Claude API

Ask Claude Opus 4.7 to propose 4-8 features that:
- **vary** meaningfully across the 4 candidates (otherwise useless for discrimination),
- **plausibly drive preference** (the kind of dimension a human cares about), and
- **capture potential disagreement** (different annotators reasonably weight them differently).

Each feature gets a 1-5 Likert score per response. Output is validated against a Pydantic schema. The system prompt carries `cache_control` so when this is later run for thousands of sets the instructions are read from cache at ~10% input cost.

In [13]:
# Pydantic schema. Anthropic SDK auto-strips numerical constraints client-side; the
# Likert range is enforced via Pydantic validation after parsing.
class FeatureScores(BaseModel):
    response_1: int = Field(..., ge=1, le=5, description="1-5 Likert score for response_1")
    response_2: int = Field(..., ge=1, le=5, description="1-5 Likert score for response_2")
    response_3: int = Field(..., ge=1, le=5, description="1-5 Likert score for response_3")
    response_4: int = Field(..., ge=1, le=5, description="1-5 Likert score for response_4")


class Feature(BaseModel):
    name: str = Field(
        ...,
        description="Short snake_case identifier (e.g. 'specificity', 'tone_warmth', 'evaluative_stance').",
    )
    definition: str = Field(
        ...,
        description="One-sentence operational definition of what this feature measures.",
    )
    rationale: str = Field(
        ...,
        description="Why annotators are likely to care about this and/or disagree on its weight.",
    )
    scores: FeatureScores = Field(
        ...,
        description="1-5 Likert score for each of the 4 candidate responses on this feature.",
    )


class FeatureSet(BaseModel):
    features: list[Feature] = Field(
        ...,
        description=(
            "Between 4 and 8 features. Each feature must vary across the 4 responses "
            "(do not propose features where all four responses score identically)."
        ),
    )

In [14]:
SYSTEM_PROMPT = """\
You are helping build an interpretable feature basis for a linear preference model over 4-way response choices.

For each input choice set, propose between 4 and 8 features (orthogonal-ish dimensions of variation) that:
1. VARY meaningfully across the 4 candidate responses — a feature where all four score the same is useless.
2. Plausibly DRIVE annotator preference — the kinds of dimensions a thoughtful human would weigh when picking.
3. Capture potential DISAGREEMENT — dimensions different annotators could reasonably weight differently.

Each feature must be:
- Linearizable: scoreable on a single 1-5 Likert scale per response (1 = least, 5 = most of the property)
- Interpretable: a short snake_case name + one-sentence definition a non-expert can apply consistently
- Substantive: avoid trivial features like raw token count unless they are clearly the differentiator. Prefer dimensions like specificity, hedging, evaluative stance, structure, factual coverage, register, warmth, qualification of claims, scope of advice, etc. — pick what actually distinguishes THESE four responses.

Score every response on every feature. Use the FULL 1-5 range when the responses meaningfully span the dimension; do not bunch all four on the same value.

When annotator feedback is provided, USE IT to ground feature selection in what humans actually noticed and articulated as their reason for choosing — not just what an LLM might find theoretically interesting.
"""


def build_user_message(
    prompt_text: str,
    responses: dict[str, str],
    choice_counts: dict,
    feedback_sample: pd.DataFrame,
) -> str:
    """Render the choice-set context into a single user message."""
    counts_block = "\n".join(
        f"  {code}: {cnt}" for code, cnt in sorted(choice_counts.items(), key=lambda kv: str(kv[0]))
    )
    resp_block = "\n\n".join(f"[{k}]\n{v}" for k, v in responses.items())
    if len(feedback_sample) == 0:
        fb_block = "  (no annotator feedback available)"
    else:
        fb_lines = []
        for i, row in feedback_sample.iterrows():
            text = str(row["first_turn_feedback"])
            if len(text) > 500:
                text = text[:500] + " ..."
            fb_lines.append(f"  [{i+1}] (chose {row['chosen_response']}) {text}")
        fb_block = "\n".join(fb_lines)

    return f"""\
PROMPT:
{prompt_text}

ANNOTATOR CHOICE DISTRIBUTION (which response each annotator picked):
{counts_block}

CANDIDATE RESPONSES:

{resp_block}

ANNOTATOR FEEDBACK (rationales annotators wrote when they picked):
{fb_block}

Propose 4-8 features per the instructions, and score each response on each feature."""


user_message = build_user_message(prompt_text, responses, choice_counts, feedback_sample_df)
print(f"User message length: {len(user_message):,} chars")

User message length: 4,015 chars


In [15]:
client = anthropic.Anthropic()

response = client.messages.parse(
    model="claude-opus-4-7",
    max_tokens=16000,
    thinking={"type": "adaptive"},
    system=[
        {
            "type": "text",
            "text": SYSTEM_PROMPT,
            "cache_control": {"type": "ephemeral"},
        }
    ],
    messages=[{"role": "user", "content": user_message}],
    output_format=FeatureSet,
)

usage = response.usage
print(
    f"Usage: input={usage.input_tokens}  output={usage.output_tokens}  "
    f"cache_read={usage.cache_read_input_tokens}  cache_create={usage.cache_creation_input_tokens}"
)
feature_set: FeatureSet = response.parsed_output
print(f"Got {len(feature_set.features)} features.")

Usage: input=2961  output=1035  cache_read=0  cache_create=0
Got 8 features.


## 5. Display the (responses × features) score matrix

In [16]:
rows = []
for f in feature_set.features:
    rows.append(
        {
            "feature": f.name,
            "definition": f.definition,
            "rationale": f.rationale,
            "response_1": f.scores.response_1,
            "response_2": f.scores.response_2,
            "response_3": f.scores.response_3,
            "response_4": f.scores.response_4,
        }
    )
features_df = pd.DataFrame(rows)

print("Feature definitions and rationales:")
display(features_df[["feature", "definition", "rationale"]])

Feature definitions and rationales:


,feature,definition,rationale
0,number_of_routes_suggested,How many distinct scenic roads or routes the response names for the user.,"Several annotators explicitly preferred responses that offered more options (D named most, A and B offered two, C focused on one)."
1,descriptive_richness,"The depth of description of what the user will see or experience along each route, beyond just naming roads.",Annotators choosing C and D praised the richness of detail and description of viewpoints/landscapes; A was praised for being concise.
2,practical_safety_guidance,Whether the response includes practical caveats like checking road/weather conditions or safety advice.,Annotator 5 specifically valued C's warning to check road and weather conditions; others did not include this.
3,conciseness,How brief and to-the-point the response is without unnecessary elaboration.,Annotator 7 explicitly preferred A for being the shortest while still useful; longer responses may feel verbose to some.
4,choice_framing,"How explicitly the response frames alternatives around different user preferences (e.g., beach vs. mountain).",Annotator 6 chose B specifically because it cleanly separated proposals by user preference type.
5,factual_plausibility,"How plausible/accurate the road designations and geographic claims appear (the actual Rio-Santos highway is BR-101/SP-055, not SP-150 or SP-122).","Some responses invent or misuse highway codes (SP-150, SP-122, SP-101 through Serra do Mar), which a knowledgeable annotator might penalize."
6,specific_landmarks_mentioned,"Degree to which specific named viewpoints, beaches, or landmarks are cited beyond just road numbers.",C stands out for naming Mirante de Santos and Praia de Pernambuco; others stay at the road-name level.
7,conversational_warmth,"How friendly, engaging, and conversational the tone is versus dry or list-like.",Annotator 8 valued A's conversational pleasant tone; annotator 4 noted D's perfect tone; this varies across responses.


In [17]:
score_matrix = features_df.set_index("feature")[
    ["response_1", "response_2", "response_3", "response_4"]
]
score_matrix["variance"] = score_matrix.var(axis=1)

print("Score matrix (rows = features, cols = responses; variance flags features that vary across responses):")
display(score_matrix.sort_values("variance", ascending=False))

# Cross-check against empirical choices: which response 'wins' on each feature?
winners = score_matrix[["response_1", "response_2", "response_3", "response_4"]].idxmax(axis=1)
print("\nPer-feature top response (this is the response that scores highest on each feature):")
display(winners.to_frame("top_response"))

# Empirical favorite — which response did annotators actually pick most often?
emp_top = max(choice_counts.items(), key=lambda kv: (kv[1], str(kv[0])))
print(
    f"\nEmpirical favorite among annotators: {emp_top[0]} ({emp_top[1]} of {sum(v for v in choice_counts.values() if isinstance(v, int))} votes)."
    " Compare against the per-feature top-response column above to see which features 'agree' with annotators."
)

Score matrix (rows = features, cols = responses; variance flags features that vary across responses):


,response_1,response_2,response_3,response_4,variance
feature,,,,,
practical_safety_guidance,1,1,5,1,4.000000
choice_framing,2,5,1,3,2.916667
specific_landmarks_mentioned,2,3,5,2,2.000000
descriptive_richness,2,3,5,4,1.666667
conciseness,5,4,2,3,1.666667
factual_plausibility,2,1,4,3,1.666667
number_of_routes_suggested,3,3,2,5,1.583333
conversational_warmth,4,4,5,3,0.666667



Per-feature top response (this is the response that scores highest on each feature):


,top_response
feature,
number_of_routes_suggested,response_4
descriptive_richness,response_3
practical_safety_guidance,response_3
conciseness,response_1
choice_framing,response_2
factual_plausibility,response_3
specific_landmarks_mentioned,response_3
conversational_warmth,response_3



Empirical favorite among annotators: response_d (4 of 15 votes). Compare against the per-feature top-response column above to see which features 'agree' with annotators.


## 6. Notes for scaling this up

- **Prompt cache.** The system prompt is `ephemeral`-cached (5-minute TTL). Subsequent per-set calls within 5 minutes read it at ~10% input cost. For long-running batch runs, switch to `"ttl": "1h"` once you've decided the system prompt is frozen.
- **Local vs. shared basis.** Each call returns features *local to that choice set*. To get a shared basis across all retained sets, run this for every set, then do a second pass that canonicalizes features (cluster by `name + definition`, optionally by embedding similarity) into a unified vocabulary, and re-score every response on the canonical features.
- **Drop low-variance features.** A feature whose 4 scores are nearly identical can't help discriminate choices in a linear-preference model — drop or merge with related dimensions during canonicalization.
- **Noise.** Single-call Likert scores from one LLM are noisy. For production, sample multiple feature sets per choice set (e.g., 3 calls with different seeds/orderings) and reconcile, or score with a separate verifier model.
- **Cost control.** If running for thousands of sets, switch to the Batches API (50% cost) — same `messages.parse()` shape works inside batch requests.